# μP coordinate check

Compare corresponding hidden activations and one-update parameter deltas at multiple widths. A passing check is a prerequisite for treating the proxy LR sweep as a μTransfer result. Run on the MLX environment.

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten
from llm_training.models.config import GPTConfig
from llm_training.models.gpt_mlx import GPT
from llm_training.pretraining_mlx.train import make_optimizer, TrainConfig

mx.random.seed(7)
vocab_size, context, layers, base_width = 128, 32, 2, 32
tokens = mx.arange(8 * context).reshape(8, context) % vocab_size
targets = mx.concatenate([tokens[:, 1:], tokens[:, :1]], axis=1)

In [ ]:
def inspect_width(width):
    config = GPTConfig(vocab_size=vocab_size, context_length=context, n_layers=layers,
                       n_heads=4, d_model=width, parametrization='mup',
                       mup_base_width=base_width)
    model = GPT(config)
    activations = {}
    def observe(name, values):
        activations[name] = mx.sqrt(mx.mean(mx.square(values.astype(mx.float32))))
    loss, gradients = nn.value_and_grad(model, lambda x, y: model(x, y, activation_observer=observe)[1])(tokens, targets)
    mx.eval(loss, gradients, activations)
    before = dict(tree_flatten(model.trainable_parameters()))
    optimizer = make_optimizer(TrainConfig(steps=1, batch_size=8, learning_rate=3e-4, weight_decay=0.0), config)
    optimizer.update(model, gradients)
    after = dict(tree_flatten(model.trainable_parameters()))
    mx.eval(after)
    deltas = {name: mx.sqrt(mx.mean(mx.square((after[name] - before[name]).astype(mx.float32))))
              for name in before}
    mx.eval(deltas)
    return {'width': width, 'loss': loss.item(),
            'activation_rms': {name: value.item() for name, value in activations.items()},
            'update_rms': {name: value.item() for name, value in deltas.items()}}

reports = [inspect_width(width) for width in (64, 128)]
reports

## Inspect

Compare each layer's activation RMS and corresponding matrix/vector update RMS across widths. Large systematic drift means the implementation needs correction before using the sweep's transferred LR. This first check is an empirical diagnostic, not a formal proof of μP.